# RQ-NAC Compression on EuroSAT (Google Colab)

Train RQ-VAE models at quantization depths **1** (VQ-VAE baseline), **4**, and **8** on the EuroSAT satellite dataset, then run N-gram Arithmetic Coding (NAC) compression on the extracted codes.

**Architecture:** 64×64 input → 8×8 latent (3 downsamples, ch_mult=[1,2,2,4]) → codebook K=2048, D=256

**Setup:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells in order
3. Each depth trains for 150 epochs (~30-40 min per depth on T4)
4. Results and checkpoints are saved to Google Drive

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clean any previous clone and re-clone fresh
!rm -rf /content/research
!git clone https://github.com/HemanthSud/OEC-Image-Processing-.git /content/research
%cd /content/research

# Install dependencies
!pip install -q omegaconf lpips PyYAML tqdm

# Generate the dataset split indices file
# split_indices.py uses __file__ to find EuroSAT_RGB/ and writes eurosat_split_indices.pt
# next to itself, so we just run it from the repo root
!python split_indices.py

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('No GPU! Go to Runtime -> Change runtime type -> T4 GPU')

# Verify split file was generated
import os
split_file = '/content/research/eurosat_split_indices.pt'
assert os.path.isfile(split_file), f'{split_file} not found! Re-run the setup cell above.'
indices = torch.load(split_file, weights_only=True)
print(f'Split indices OK: train={len(indices["train"])}, val={len(indices["val"])}, test={len(indices["test"])}')


## 2. Verify Dataset & Load Imports

In [ ]:
import os, sys, math, shutil, time
import torch
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from omegaconf import OmegaConf
import yaml

# Verify dataset exists
eurosat_dir = '/content/research/EuroSAT_RGB'
assert os.path.isdir(eurosat_dir), f'{eurosat_dir} not found!'
classes = sorted([c for c in os.listdir(eurosat_dir) if os.path.isdir(os.path.join(eurosat_dir, c))])
total = sum(len(os.listdir(os.path.join(eurosat_dir, c))) for c in classes)
print(f'EuroSAT: {len(classes)} classes, {total} images')

# Add rq-vae package to Python path (contains rqvae/ package)
rqvae_root = '/content/research/rq-vae'
if rqvae_root not in sys.path:
    sys.path.insert(0, rqvae_root)

# Add nac directory to Python path (contains ngram.py, arithmetic_coding.py)
nac_root = '/content/research/nac'
if nac_root not in sys.path:
    sys.path.insert(0, nac_root)

# RQ-VAE imports
from rqvae.models.rqvae.rqvae import RQVAE
from rqvae.img_datasets.eurosat import EuroSAT
from rqvae.img_datasets.transforms import create_transforms
from rqvae.losses.vqgan.lpips import LPIPS
from rqvae.losses.vqgan.discriminator import NLayerDiscriminator, weights_init
from rqvae.losses.vqgan.gan_loss import hinge_d_loss, vanilla_g_loss

# NAC imports
from ngram import NGramModel
from arithmetic_coding import ArithmeticEncoder

device = torch.device('cuda')
print(f'\nAll imports OK. Device: {device}')

## 3. Define Training & NAC Functions

In [ ]:
EUROSAT_DIR = '/content/research/EuroSAT_RGB'
SPLIT_INDICES = '/content/research/eurosat_split_indices.pt'

def calculate_adaptive_weight(nll_loss, g_loss, last_layer):
    nll_grads = torch.autograd.grad(nll_loss, last_layer, retain_graph=True)[0]
    g_grads = torch.autograd.grad(g_loss, last_layer, retain_graph=True)[0]
    d_weight = torch.norm(nll_grads) / (torch.norm(g_grads) + 1e-4)
    return torch.clamp(d_weight, 0.0, 1e4).detach()


def train_rqvae(config_path, output_dir, epochs=150, batch_size=32, lr=4e-5):
    """Train a single RQ-VAE model from a config file. Returns model, loaders, and history."""
    os.makedirs(output_dir, exist_ok=True)

    with open(config_path) as f:
        config = OmegaConf.create(yaml.load(f, Loader=yaml.FullLoader))
    config.experiment.epochs = epochs
    config.experiment.batch_size = batch_size
    config.optimizer.init_lr = lr
    OmegaConf.save(config, os.path.join(output_dir, 'config.yaml'))

    depth = config.arch.hparams.code_shape[-1]
    print(f'\n{"="*60}')
    print(f'TRAINING DEPTH={depth} | {config_path}')
    print(f'{"="*60}')

    # Datasets (use absolute paths so CWD doesn't matter)
    dataset_cfg = OmegaConf.create({'transforms': config.dataset.transforms})
    transforms_trn = create_transforms(dataset_cfg, split='train', is_eval=False)
    transforms_val = create_transforms(dataset_cfg, split='val', is_eval=True)
    dataset_trn = EuroSAT(EUROSAT_DIR, split='train', transform=transforms_trn,
                           split_indices_path=SPLIT_INDICES)
    dataset_val = EuroSAT(EUROSAT_DIR, split='val', transform=transforms_val,
                           split_indices_path=SPLIT_INDICES)
    loader_trn = DataLoader(dataset_trn, batch_size=batch_size, shuffle=True,
                            num_workers=2, pin_memory=True, drop_last=True)
    loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
    print(f'Train: {len(dataset_trn)}, Val: {len(dataset_val)}')

    # Model
    hps = dict(config.arch.hparams)
    ddconfig = dict(config.arch.ddconfig)
    model = RQVAE(**hps, ddconfig=ddconfig,
                  checkpointing=config.arch.get('checkpointing', False)).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'Parameters: {n_params:.2f}M')

    # Discriminator
    gc = config.gan
    disc = NLayerDiscriminator(
        input_nc=gc.disc.arch.in_channels, n_layers=gc.disc.arch.num_layers,
        use_actnorm=gc.disc.arch.use_actnorm, ndf=gc.disc.arch.ndf,
    ).apply(weights_init).to(device)

    # Losses & optimizers
    lpips_loss = LPIPS().to(device).eval()
    pw = gc.loss.perceptual_weight
    dw = gc.loss.disc_weight
    gan_start = gc.loss.disc_start

    betas = tuple(config.optimizer.betas)
    opt_g = torch.optim.Adam(model.parameters(), lr=lr, betas=betas)
    opt_d = torch.optim.Adam(disc.parameters(), lr=lr, betas=betas)

    total_steps = epochs * len(loader_trn)
    warmup_steps = config.optimizer.warmup.epoch * len(loader_trn)
    def lr_lambda(step):
        if step < warmup_steps:
            return max(step / max(warmup_steps, 1), 1e-2)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return max(0.5 * (1 + math.cos(math.pi * progress)), 1e-2)
    sched_g = torch.optim.lr_scheduler.LambdaLR(opt_g, lr_lambda)
    sched_d = torch.optim.lr_scheduler.LambdaLR(opt_d, lr_lambda)

    # Training loop
    best_val = float('inf')
    train_hist, val_hist = [], []

    for epoch in range(epochs):
        model.train(); disc.train()
        use_gan = (epoch >= gan_start)
        epoch_recon = 0

        pbar = tqdm(loader_trn, desc=f'D{depth} Ep{epoch+1}/{epochs}', leave=False)
        for imgs, _ in pbar:
            imgs = imgs.to(device)
            opt_g.zero_grad()
            recon, ql, codes = model(imgs)
            lo = model.compute_loss(recon, ql, codes, xs=imgs)
            loss_recon = lo['loss_recon']
            loss_pcpt = lpips_loss(imgs, recon)

            if use_gan:
                lf, _ = disc(recon.contiguous(), None)
                loss_gen = vanilla_g_loss(lf)
                gw = calculate_adaptive_weight(loss_recon + pw * loss_pcpt,
                                               loss_gen, model.get_last_layer())
            else:
                loss_gen = torch.tensor(0., device=device)
                gw = torch.tensor(0., device=device)

            loss_g = lo['loss_total'] + pw * loss_pcpt + gw * dw * loss_gen
            loss_g.backward()
            opt_g.step(); sched_g.step()

            if use_gan:
                opt_d.zero_grad()
                lf, lr_ = disc(recon.detach(), imgs.detach())
                (dw * hinge_d_loss(lr_, lf)).backward()
                opt_d.step(); sched_d.step()

            epoch_recon += loss_recon.item()
            pbar.set_postfix(recon=f'{loss_recon.item():.4f}')

        avg_recon = epoch_recon / len(loader_trn)
        train_hist.append(avg_recon)

        # Validation
        model.eval()
        vr = 0
        with torch.no_grad():
            for imgs, _ in loader_val:
                imgs = imgs.to(device)
                r, ql, c = model(imgs)
                vr += model.compute_loss(r, ql, c, xs=imgs, valid=True)['loss_recon'].item()
        avg_val = vr / max(len(loader_val), 1)
        val_hist.append(avg_val)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'  Ep{epoch+1} train_recon={avg_recon:.4f} val_recon={avg_val:.4f}')

        if avg_val < best_val:
            best_val = avg_val
            torch.save({'epoch': epoch+1, 'state_dict': model.state_dict()},
                       os.path.join(output_dir, 'best_model.pt'))

        if (epoch+1) % 25 == 0:
            torch.save({'epoch': epoch+1, 'state_dict': model.state_dict(),
                        'discriminator': disc.state_dict()},
                       os.path.join(output_dir, f'epoch{epoch+1}_model.pt'))

        if (epoch+1) % 50 == 0 or epoch == 0:
            with torch.no_grad():
                s = imgs[:8]
                sr = model(s)[0]
                grid = torch.cat([s * 0.5 + 0.5, torch.clamp(sr * 0.5 + 0.5, 0, 1)])
                torchvision.utils.save_image(
                    torchvision.utils.make_grid(grid, nrow=8),
                    os.path.join(output_dir, f'recon_ep{epoch+1:03d}.png'))

    print(f'Depth {depth} done. Best val_recon: {best_val:.4f}')
    return model, config, loader_trn, loader_val, train_hist, val_hist


def extract_codes(model, config, loader_trn, loader_val):
    """Extract RQ codes from trained model and save to nac/data/."""
    code_shape = list(config.arch.hparams.code_shape)
    H, W, D = code_shape
    code_file = f'/content/research/nac/data/codes{H}x{W}x{D}.txt'
    os.makedirs(os.path.dirname(code_file), exist_ok=True)
    if os.path.exists(code_file):
        os.remove(code_file)

    model.eval()
    n_images = 0
    with torch.no_grad():
        for name, loader in [('train', loader_trn), ('val', loader_val)]:
            for imgs, _ in tqdm(loader, desc=f'Extracting {name} codes (D={D})'):
                codes = model.get_codes(imgs.to(device)).cpu().numpy()
                with open(code_file, 'a') as f:
                    for i in range(codes.shape[0]):
                        f.write(' '.join(map(str, codes[i].reshape(-1).astype(int))) + '\n')
                n_images += codes.shape[0]

    print(f'Saved {n_images} sequences -> {code_file} ({H}x{W}x{D} = {H*W*D} codes/image)')
    return code_file, H, W, D


def run_nac(code_file, H, W, D, n_gram=2, k_smooth=0.1):
    """Run N-gram Arithmetic Coding on extracted codes. Returns results dict."""
    def readcode(fn, n=None):
        result = []
        with open(fn, 'r') as f:
            for i, line in enumerate(f):
                if n is not None and i >= n:
                    break
                line = line.strip()
                if line:
                    result.append([int(x) for x in line.split()])
        return result

    all_codes = readcode(code_file)
    n_train = int(len(all_codes) * 0.9)
    train_seqs = all_codes[:n_train]
    test_seqs = all_codes[n_train:]

    print(f'\nNAC D={D}: {n_gram}-gram, K={k_smooth}, {len(train_seqs)} train, {len(test_seqs)} test')

    ngram_model = NGramModel(n=n_gram, k=k_smooth, start_token=-1, end_token=-2)
    ngram_model.fit(train_seqs)

    os.makedirs('/content/research/nac/models', exist_ok=True)
    ngram_model.save(f'/content/research/nac/models/{n_gram}gram_eurosat_{H}x{W}x{D}.pkl')

    encoder = ArithmeticEncoder(ngram_model=ngram_model, bits=32)
    BITS_PER_CODE = 11
    RAW_IMAGE_BITS = 64 * 64 * 3 * 8

    rates, ratios, enc_t, dec_t, errors = [], [], [], [], 0
    for seq in tqdm(test_seqs, desc=f'NAC D={D}'):
        t0 = time.perf_counter()
        bits = encoder.encode(seq)
        enc_t.append((time.perf_counter() - t0) * 1000)

        t0 = time.perf_counter()
        decoded = encoder.decode(bits)
        dec_t.append((time.perf_counter() - t0) * 1000)

        if decoded != seq:
            errors += 1
        rates.append(len(bits) / (len(seq) * BITS_PER_CODE))
        ratios.append(RAW_IMAGE_BITS / len(bits))

    avg_rate = sum(rates) / len(rates)
    avg_ratio = sum(ratios) / len(ratios)
    avg_bpp = sum(len(encoder.encode(s)) for s in test_seqs) / (len(test_seqs) * 64 * 64)

    print(f'  Compression rate (vs uncompressed codes): {avg_rate:.2%}')
    print(f'  Compression ratio (vs raw RGB): {avg_ratio:.1f}x')
    print(f'  Bits per pixel (bpp): {avg_bpp:.3f}')
    print(f'  Errors: {errors}/{len(test_seqs)}')

    return {'depth': D, 'rate': avg_rate, 'ratio': avg_ratio, 'bpp': avg_bpp,
            'errors': errors, 'n_test': len(test_seqs)}

print('Functions defined.')

In [ ]:
## 4. Train All Depths (1, 4, 8) + Extract Codes + Run NAC

In [ ]:
# Depths to evaluate: 1 (VQ-VAE baseline), 4, 8
DEPTHS = [1, 4, 8]
EPOCHS = 150
BATCH_SIZE = 32
LR = 4e-5

configs = {
    1: '/content/research/rq-vae/configs/eurosat/stage1/eurosat-rqvae-8x8x1.yaml',
    4: '/content/research/rq-vae/configs/eurosat/stage1/eurosat-rqvae-8x8x4.yaml',
    8: '/content/research/rq-vae/configs/eurosat/stage1/eurosat-rqvae-8x8x8.yaml',
}

all_results = {}
all_histories = {}

for depth in DEPTHS:
    out_dir = f'/content/research/rq-vae/output/eurosat_d{depth}'
    model, config, loader_trn, loader_val, th, vh = train_rqvae(
        configs[depth], out_dir, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR)

    code_file, H, W, D = extract_codes(model, config, loader_trn, loader_val)
    nac_result = run_nac(code_file, H, W, D)

    all_results[depth] = nac_result
    all_histories[depth] = {'train': th, 'val': vh}

    # Free GPU memory before next depth
    del model
    torch.cuda.empty_cache()

print('\n' + '='*60)
print('ALL DEPTHS COMPLETE')
print('='*60)

In [ ]:
## 5. Comparison: Results Table

In [ ]:
print(f'{"Depth":<8} {"Codes/img":<12} {"NAC Rate":<12} {"Ratio vs RGB":<14} {"BPP":<10} {"Errors":<8}')
print('-' * 64)
for d in DEPTHS:
    r = all_results[d]
    codes_per_img = 8 * 8 * d
    print(f'{d:<8} {codes_per_img:<12} {r["rate"]:<12.2%} {r["ratio"]:<14.1f}x {r["bpp"]:<10.3f} {r["errors"]}/{r["n_test"]}')

## 6. Training Curves Comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for d in DEPTHS:
    label = f'D={d}' + (' (VQ-VAE)' if d == 1 else '')
    axes[0].plot(all_histories[d]['train'], label=label)
    axes[1].plot(all_histories[d]['val'], label=label)

axes[0].set_title('Train Reconstruction Loss')
axes[1].set_title('Val Reconstruction Loss')
for ax in axes:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('RQ-VAE Training: Depth Comparison on EuroSAT', fontsize=14)
plt.tight_layout()
plt.savefig('/content/research/rq-vae/output/depth_comparison_curves.png', dpi=150)
plt.show()

## 7. Compression Comparison Bar Chart

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

depths_list = [str(d) for d in DEPTHS]
bpps = [all_results[d]['bpp'] for d in DEPTHS]
ratios = [all_results[d]['ratio'] for d in DEPTHS]
rates = [all_results[d]['rate'] * 100 for d in DEPTHS]

axes[0].bar(depths_list, bpps, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[0].set_title('Bits Per Pixel (BPP)')
axes[0].set_xlabel('RQ Depth')
axes[0].set_ylabel('BPP')
for i, v in enumerate(bpps):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

axes[1].bar(depths_list, ratios, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[1].set_title('Compression Ratio vs Raw RGB')
axes[1].set_xlabel('RQ Depth')
axes[1].set_ylabel('Ratio (x)')
for i, v in enumerate(ratios):
    axes[1].text(i, v + 0.5, f'{v:.1f}x', ha='center', fontsize=10)

axes[2].bar(depths_list, rates, color=['#2196F3', '#4CAF50', '#FF9800'])
axes[2].set_title('NAC Rate vs Uncompressed Codes')
axes[2].set_xlabel('RQ Depth')
axes[2].set_ylabel('Rate (%)')
for i, v in enumerate(rates):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)

plt.suptitle('RQ-NAC Compression Results on EuroSAT', fontsize=14)
plt.tight_layout()
plt.savefig('/content/research/rq-vae/output/compression_comparison.png', dpi=150)
plt.show()

## 8. Visualize Reconstructions (All Depths)

In [ ]:
DEPTHS = [1, 4, 8]

# Load a val batch for comparison
dataset_cfg = OmegaConf.create({'transforms': {'type': 'eurosat'}})
transforms_val = create_transforms(dataset_cfg, split='val', is_eval=True)
dataset_val = EuroSAT('/content/research/EuroSAT_RGB', split='val', transform=transforms_val,
                       split_indices_path='/content/research/eurosat_split_indices.pt')
vis_loader = DataLoader(dataset_val, batch_size=8, shuffle=False, num_workers=2)
vis_imgs = next(iter(vis_loader))[0].to(device)

n_depths = len(DEPTHS)
fig, axes = plt.subplots(n_depths + 1, 8, figsize=(20, 3 * (n_depths + 1)))

# Row 0: originals
imgs_vis = (vis_imgs.cpu() * 0.5 + 0.5).clamp(0, 1)
for i in range(8):
    axes[0, i].imshow(imgs_vis[i].permute(1, 2, 0).numpy())
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=12, rotation=0, labelpad=60, va='center')

# Rows 1+: reconstructions per depth
for row, d in enumerate(DEPTHS, 1):
    out_dir = f'/content/research/rq-vae/output/eurosat_d{d}'
    with open(os.path.join(out_dir, 'config.yaml')) as f:
        cfg = OmegaConf.create(yaml.load(f, Loader=yaml.FullLoader))

    hps = dict(cfg.arch.hparams)
    ddconfig = dict(cfg.arch.ddconfig)
    m = RQVAE(**hps, ddconfig=ddconfig, checkpointing=False).to(device)
    ckpt = torch.load(os.path.join(out_dir, 'best_model.pt'), weights_only=False)
    m.load_state_dict(ckpt['state_dict'])
    m.eval()

    with torch.no_grad():
        recon = m(vis_imgs)[0]
    recon_vis = (recon.cpu() * 0.5 + 0.5).clamp(0, 1)

    for i in range(8):
        axes[row, i].imshow(recon_vis[i].permute(1, 2, 0).numpy())
        axes[row, i].axis('off')
    label = f'D={d}' + (' (VQ-VAE)' if d == 1 else '')
    axes[row, 0].set_ylabel(label, fontsize=12, rotation=0, labelpad=60, va='center')
    del m

plt.suptitle('Reconstruction Comparison Across Depths', fontsize=14)
plt.tight_layout()
plt.savefig('/content/research/rq-vae/output/recon_depth_comparison.png', dpi=150)
plt.show()

## 9. Save All Results to Google Drive

In [ ]:
DEPTHS = [1, 4, 8]
drive_output = '/content/drive/MyDrive/rqnac_eurosat_results'
os.makedirs(drive_output, exist_ok=True)

# Save per-depth outputs
for d in DEPTHS:
    src_dir = f'/content/research/rq-vae/output/eurosat_d{d}'
    dst_dir = os.path.join(drive_output, f'depth_{d}')
    os.makedirs(dst_dir, exist_ok=True)
    for f in os.listdir(src_dir):
        src = os.path.join(src_dir, f)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(dst_dir, f))

    # Copy NAC codes
    code_src = f'/content/research/nac/data/codes8x8x{d}.txt'
    if os.path.exists(code_src):
        shutil.copy2(code_src, os.path.join(dst_dir, f'codes8x8x{d}.txt'))

# Save comparison plots
for f in ['depth_comparison_curves.png', 'compression_comparison.png', 'recon_depth_comparison.png']:
    src = f'/content/research/rq-vae/output/{f}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(drive_output, f))

print(f'All results saved to: {drive_output}')
!ls -la {drive_output}/